In [1]:
# ==========================================
# CELL 1: Environment Setup & Dependencies
# ==========================================
import os
import time
import random
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

# Set fixed seeds for strict reproducibility
random.seed(42)
torch.manual_seed(42)

# Hardware acceleration setup (CUDA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Compute Device: {device}")

💻 Compute Device: cuda


In [2]:
# ==========================================
# CELL 2: Clinical Data Curation
# ==========================================
# Define dataset paths
chemin_csv = '/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/clinical_data(labels).csv'
chemin_wsis = '/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/WSIs'

# Load clinical metadata
df_clinique = pd.read_csv(chemin_csv)

# Format survival labels for Binary Cross Entropy (1 -> 0: Alive, 2 -> 1: Dead)
df_clinique['vital_status'] = df_clinique['vital_status'].replace({1: 0, 2: 1})

# Feature selection: Core variables + TNM/Stage pathology metrics
colonnes_de_base = ['bcr_patient_barcode', 'Time', 'vital_status', 'age_at_initial_pathologic_diagnosis']
colonnes_tnm_stage = [col for col in df_clinique.columns if 'pathologic_' in col]
toutes_les_colonnes = list(set(colonnes_de_base + colonnes_tnm_stage)) # Merge and remove duplicates

# Create clean dataframe
df_clean = df_clinique[toutes_les_colonnes].copy()
df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()].copy()

print(f"✅ Clinical data curated. Final Shape: {df_clean.shape}")
print("Label Distribution (0=Alive, 1=Dead):")
print(df_clean['vital_status'].value_counts())

✅ Clinical data curated. Final Shape: (1063, 18)
Label Distribution (0=Alive, 1=Dead):
vital_status
0    914
1    149
Name: count, dtype: int64


In [3]:
# ==========================================
# CELL 3: Vision Pipeline & Custom Dataset
# ==========================================
# Standard ImageNet preprocessing pipeline
transformations = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BRCA_WSI_Dataset(Dataset):
    """Custom PyTorch Dataset for loading Gigapixel WSI patches."""
    def __init__(self, df, dossier_racine, transform=None, max_patches=10):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        print(f"🔍 Indexing virtual slides (Max {max_patches} patches/patient)...")
        patients_trouves = 0
        
        for index, row in df.iterrows():
            patient_id = row['bcr_patient_barcode'] 
            label = row['vital_status'] 
            dossier_patient = os.path.join(dossier_racine, patient_id)
            
            if os.path.exists(dossier_patient):
                patients_trouves += 1
                images = [f for f in os.listdir(dossier_patient) if f.endswith(('.jpg', '.png'))]
                
                for img in images[:max_patches]:
                    self.image_paths.append(os.path.join(dossier_patient, img))
                    # Direct append: label is already converted to integer (0 or 1) in Cell 2
                    self.labels.append(int(label))
                    
        print(f"✅ Indexing complete: {len(self.image_paths)} patches linked to {patients_trouves} patients.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.float32)

In [4]:
# ==========================================
# CELL 4: Patient-Level Validation Split
# ==========================================
# Extract unique patient identifiers
patients_uniques = df_clean['bcr_patient_barcode'].unique().tolist()
random.shuffle(patients_uniques) # Seed 42 ensures the same shuffle every time

# Define strict 80/20 boundaries
taille_train_patients = int(0.8 * len(patients_uniques))
patients_train = patients_uniques[:taille_train_patients]
patients_val = patients_uniques[taille_train_patients:]

# Route patients into train and validation subsets
df_train = df_clean[df_clean['bcr_patient_barcode'].isin(patients_train)]
df_val = df_clean[df_clean['bcr_patient_barcode'].isin(patients_val)]

# Instantiate Datasets
train_dataset = BRCA_WSI_Dataset(df_train, chemin_wsis, transform=transformations)
val_dataset = BRCA_WSI_Dataset(df_val, chemin_wsis, transform=transformations)

# Build continuous DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"🧮 Cohort Split Secured: {len(patients_train)} Train Patients | {len(patients_val)} Validation Patients.")

🔍 Indexing virtual slides (Max 10 patches/patient)...
✅ Indexing complete: 7950 patches linked to 795 patients.
🔍 Indexing virtual slides (Max 10 patches/patient)...
✅ Indexing complete: 1950 patches linked to 195 patients.
🧮 Cohort Split Secured: 850 Train Patients | 213 Validation Patients.


In [5]:
# ==========================================
# CELL 5: Deep Learning Architecture
# ==========================================
print("🧠 Initializing Pre-trained ResNet50...")

# Load Transfer Learning weights
modele_vision = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze base convolutional layers to retain standard feature extraction
for param in modele_vision.parameters():
    param.requires_grad = False

# Modify final classification layer for single-node risk scoring
modele_vision.fc = nn.Linear(modele_vision.fc.in_features, 1)

# Push model to GPU
modele_vision = modele_vision.to(device)

# Define Loss function (Binary Cross Entropy) and Optimizer
critere = nn.BCEWithLogitsLoss()
optimiseur = optim.Adam(modele_vision.fc.parameters(), lr=0.001)

print("✅ Model, Criterion, and Optimizer successfully assembled.")

🧠 Initializing Pre-trained ResNet50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 214MB/s]


✅ Model, Criterion, and Optimizer successfully assembled.


In [6]:
# ==========================================
# CELL 6: Training & Evaluation Loop
# ==========================================
epochs = 3 
print("Launching Clinical Training Pipeline...")
start_time = time.time()

for epoch in range(epochs):
    
    # --- 1. TRAINING PHASE ---
    modele_vision.train()
    train_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.unsqueeze(1).to(device)
        
        optimiseur.zero_grad()
        predictions = modele_vision(images)
        perte = critere(predictions, labels)
        
        perte.backward()
        optimiseur.step()
        train_loss += perte.item()
        
    avg_train_loss = train_loss / len(train_loader)

    # --- 2. VALIDATION PHASE ---
    modele_vision.eval()
    val_loss = 0.0
    predictions_correctes = 0
    total_val = 0
    
    with torch.no_grad(): 
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.unsqueeze(1).to(device)
            
            predictions = modele_vision(images)
            perte = critere(predictions, labels)
            val_loss += perte.item()
            
            # Apply sigmoid for probability calculation
            probabilites = torch.sigmoid(predictions)
            predictions_finales = (probabilites > 0.5).float()
            
            predictions_correctes += (predictions_finales == labels).sum().item()
            total_val += labels.size(0)
            
    avg_val_loss = val_loss / len(val_loader)
    accuracy = (predictions_correctes / total_val) * 100
    
    # --- 3. EPOCH METRICS ---
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Accuracy: {accuracy:.2f}%")

temps_ecoule = time.time() - start_time
print(f"\n🏁 Process completed in {temps_ecoule/60:.2f} minutes.")

Launching Clinical Training Pipeline...
Epoch 1/3 | Train Loss: 0.3934 | Val Loss: 0.4461 | Accuracy: 82.10%
Epoch 2/3 | Train Loss: 0.3613 | Val Loss: 0.4424 | Accuracy: 82.46%
Epoch 3/3 | Train Loss: 0.3490 | Val Loss: 0.4430 | Accuracy: 82.62%

🏁 Process completed in 9.17 minutes.


In [ ]:
# ==========================================
# CELL 7: Sauvegarde de l'Expert Visuel
# ==========================================
chemin_sauvegarde = "resnet50_vision_expert.pth"

# On sauvegarde uniquement les "poids" (les connaissances apprises) du modèle
torch.save(modele_vision.state_dict(), chemin_sauvegarde)